In [1]:
import kagglehub
path = kagglehub.dataset_download("ahmedesso/brest-cancer")

100%|██████████| 48.6k/48.6k [00:00<00:00, 30.5MB/s]

Extracting files...


In [2]:
import os

# List the contents of the downloaded path
print(f"Contents of {path}:")
for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Contents of /root/.cache/kagglehub/datasets/ahmedesso/brest-cancer/versions/1:
/root/.cache/kagglehub/datasets/ahmedesso/brest-cancer/versions/1/data.csv


In [3]:
import pandas as pd
import os

# Construct the full path to the data.csv file
data_file_path = os.path.join(path, 'data.csv')

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(data_file_path)

# Display the first few rows of the DataFrame
print("DataFrame Head:")
print(df.head())

# Display information about the DataFrame, including data types and non-null values
print("\nDataFrame Info:")
df.info()

DataFrame Head:
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_w

In [4]:
# Drop the 'id' column as it's not useful for prediction
df = df.drop('id', axis=1)

# Drop the 'Unnamed: 32' column as it contains all NaN values
df = df.drop('Unnamed: 32', axis=1)

# Encode the 'diagnosis' column (M=1, B=0)
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Display the first few rows of the cleaned DataFrame and its info to confirm changes
print("\nDataFrame after cleaning and encoding:")
print(df.head())
print("\nDataFrame Info after cleaning:")
df.info()


DataFrame after cleaning and encoding:
   diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0          1        17.99         10.38          122.80     1001.0   
1          1        20.57         17.77          132.90     1326.0   
2          1        19.69         21.25          130.00     1203.0   
3          1        11.42         20.38           77.58      386.1   
4          1        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   symmetry_mean  ...  radius_worst  texture_worst  perimeter_worst  \
0    

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Separate features (X) and target (y)
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Display the shapes of the datasets to confirm
print(f"Shape of X_train_scaled: {X_train_scaled.shape}")
print(f"Shape of X_test_scaled: {X_test_scaled.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# Display the first few rows of scaled training data (optional, for verification)
print("\nFirst 5 rows of X_train_scaled:\n", X_train_scaled[:5])

Shape of X_train_scaled: (455, 30)
Shape of X_test_scaled: (114, 30)
Shape of y_train: (455,)
Shape of y_test: (114,)

First 5 rows of X_train_scaled:
 [[ 5.18558727e-01  8.91825791e-01  4.24631702e-01  3.83925436e-01
  -9.74743706e-01 -6.89771505e-01 -6.88586446e-01 -3.98175254e-01
  -1.03915470e+00 -8.25056321e-01 -1.09317755e-01 -5.59755400e-02
  -2.10096206e-01 -1.59132582e-02 -1.00518399e+00 -9.11941990e-01
  -6.62815884e-01 -6.52561081e-01 -7.01889114e-01 -2.75393571e-01
   5.79797697e-01  1.31324246e+00  4.66908134e-01  4.45982711e-01
  -5.96154777e-01 -6.34722227e-01 -6.10227299e-01 -2.35743918e-01
   5.45663235e-02  2.18367276e-02]
 [-5.16364088e-01 -1.63971029e+00 -5.41348716e-01 -5.42961327e-01
   4.76219058e-01 -6.31833818e-01 -6.04281166e-01 -3.03074908e-01
   5.21543093e-01 -4.54522896e-01 -6.04377961e-01 -1.00104604e+00
  -5.85429002e-01 -4.93453793e-01  4.03212009e-01 -7.68173276e-01
  -4.79187222e-01  1.14508478e-01 -1.42950761e-01 -5.77397732e-01
  -5.82458953e-01 -1.

In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the deep learning model
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Display the model summary
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,337 (56.00 KB)

 Trainable params: 14,337 (56.00 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Fix the UserWarning by explicitly defining the input layer
model_fixed = keras.Sequential([
    keras.Input(shape=(X_train_scaled.shape[1],)), # Use keras.Input as the first layer
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model_fixed.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

print("\nModel summary after fixing UserWarning:")
model_fixed.summary()

# Train the model
history = model_fixed.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate the model on the test set
loss, accuracy = model_fixed.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


Model summary after fixing UserWarning:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 128)            │         3,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,337 (56.00 KB)

 Trainable params: 14,337 (56.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.7198 - loss: 0.5733 - val_accuracy: 0.9231 - val_loss: 0.3928
Epoch 2/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9066 - loss: 0.3483 - val_accuracy: 0.9451 - val_loss: 0.2316
Epoch 3/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9423 - loss: 0.2156 - val_accuracy: 0.9451 - val_loss: 0.1684
Epoch 4/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9478 - loss: 0.1512 - val_accuracy: 0.9560 - val_loss: 0.1411
Epoch 5/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9643 - loss: 0.1157 - val_accuracy: 0.9560 - val_loss: 0.1216
Epoch 6/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9725 - loss: 0.1009 - val_accuracy: 0.9560 - val_loss: 0.1118
Epoch 7/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9725 - loss: 0.0883 - val_accuracy: 0.9670 - val_loss: 0.1050
Epoch 8/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9753 - loss: 0.0799 - val_accuracy: 0.9670 - v

### Build a Streamlit app for prediction

To deploy our model in a user-friendly interface, we will create a Streamlit application. This app will allow users to input the required features and get real-time predictions from our deep learning model. The input fields will be conveniently located in a sidebar, and the prediction output will be visually enhanced with emojis.

In [8]:
# Install Streamlit
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 76.0 MB/s eta 0:00:00


In [9]:
import joblib

# Save the scaler and the trained model
joblib.dump(scaler, 'scaler.pkl')
model_fixed.save('breast_cancer_model.h5')

print("Scaler and model saved successfully.")

Scaler and model saved successfully.


In [ ]:
import joblib
import pandas as pd

# Assuming 'X' is available from the training data split in previous cells.
# If 'X' is not in scope, you'd need to reload df and re-create X.
# For a robust solution, feature names should also be saved explicitly.

# Calculate and save feature statistics for Streamlit sliders
feature_stats = {}
for col in X.columns:
    feature_stats[col] = {
        'min': float(X[col].min()),
        'max': float(X[col].max()),
        'mean': float(X[col].mean())
    }
joblib.dump(feature_stats, 'feature_stats.pkl')
print("Feature statistics saved to feature_stats.pkl")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from tensorflow import keras

# Load the scaler and the model
scaler = joblib.load('scaler.pkl')
model = keras.models.load_model('breast_cancer_model.h5')

st.set_page_config(page_title="Breast Cancer Prediction App 🌸", layout="wide")

st.title('Breast Cancer Prediction App 🩺')
st.markdown("--- ✨ Input patient data to predict the likelihood of Malignant (1) or Benign (0) cancer. ✨ ---")

# Load feature names and statistics
feature_stats = joblib.load('feature_stats.pkl')
feature_names = list(feature_stats.keys())

# Sidebar for user input
st.sidebar.header('Patient Data Input 👇')

# Dictionary to store input values
input_data = {}

for feature in feature_names:
    stats = feature_stats[feature]
    input_data[feature] = st.sidebar.slider(f'Input for {feature}',
                                         stats['min'],
                                         stats['max'],
                                         stats['mean'])

input_df = pd.DataFrame([input_data])

st.subheader('User Input Data 📝')
st.write(input_df)

input_scaled = scaler.transform(input_df)

prediction_proba = model.predict(input_scaled)[0][0]
prediction_class = 1 if prediction_proba >= 0.5 else 0

st.subheader('Prediction Result 📊')

if prediction_class == 1:
    st.error(f'The model predicts: Malignant 🚨 (Probability: {prediction_proba:.2f})')
    st.image('https://i.ibb.co/qN3G2yP/malignant.png', caption='Malignant Indication', width=150)
else:
    st.success(f'The model predicts: Benign ✅ (Probability: {prediction_proba:.2f})')
    st.image('https://i.ibb.co/qN3G2yP/benign.png', caption='Benign Indication', width=150)

st.markdown("--- ")
st.write("Disclaimer: This app is for educational purposes only and should not be used for medical advice. Always consult with a healthcare professional. 👩‍⚕️👨‍⚕️")

In [24]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from tensorflow import keras

# Load the scaler and the model
scaler = joblib.load('scaler.pkl')
model = keras.models.load_model('breast_cancer_model.h5')

st.set_page_config(page_title="Breast Cancer Prediction App 🌸", layout="wide")

st.title('Breast Cancer Prediction App 🩺')
st.markdown("--- ✨ Input patient data to predict the likelihood of Malignant (1) or Benign (0) cancer. ✨ ---")

# Load feature names and statistics
feature_stats = joblib.load('feature_stats.pkl')
feature_names = list(feature_stats.keys())

# Sidebar for user input
st.sidebar.header('Patient Data Input 👇')

# Dictionary to store input values
input_data = {}

for feature in feature_names:
    stats = feature_stats[feature]
    input_data[feature] = st.sidebar.slider(f'Input for {feature}',
                                         stats['min'],
                                         stats['max'],
                                         stats['mean'])

input_df = pd.DataFrame([input_data])

st.subheader('User Input Data 📝')
st.write(input_df)

input_scaled = scaler.transform(input_df)

prediction_proba = model.predict(input_scaled)[0][0]
prediction_class = 1 if prediction_proba >= 0.5 else 0

st.subheader('Prediction Result 📊')

if prediction_class == 1:
    st.error(f'The model predicts: Malignant 🚨 (Probability: {prediction_proba:.2f})')
    st.image('https://i.ibb.co/qN3G2yP/malignant.png', caption='Malignant Indication', width=150)
else:
    st.success(f'The model predicts: Benign ✅ (Probability: {prediction_proba:.2f})')
    st.image('https://i.ibb.co/qN3G2yP/benign.png', caption='Benign Indication', width=150)

st.markdown("--- ")
st.write("Disclaimer: This app is for educational purposes only and should not be used for medical advice. Always consult with a healthcare professional. 👩‍⚕️👨‍⚕️")

Overwriting app.py


Now that `app.py` has been updated, please do the following:

1.  **Stop the currently running Streamlit app** by clicking the 'Interrupt execution' button (the square stop button) next to the `!streamlit run app.py &` cell.
2.  **Rerun the Streamlit app** using the cell that contains `!nohup streamlit run app.py &` (cell `70c827d8`).
3.  **Expose the app again** using the cell that contains `!lt --port 8501` (cell `336efbb6`). This should give you a new public URL to access the updated Streamlit application.

In [ ]:
# Install localtunnel (if not already installed or if ngrok is problematic)
!npm install -g localtunnel

In [ ]:
# Run the Streamlit app in the background
# This will print a message indicating the app is running but won't provide a direct link yet.
!nohup streamlit run app.py &

In [ ]:
# Expose the Streamlit app via localtunnel
!lt --port 8501

In [10]:

import streamlit as st
import pandas as pd
import numpy as np
import joblib
from tensorflow import keras

# Load the scaler and the model
scaler = joblib.load('scaler.pkl')
model = keras.models.load_model('breast_cancer_model.h5')

st.set_page_config(page_title="Breast Cancer Prediction App 🌸", layout="wide")

st.title('Breast Cancer Prediction App 🩺')
st.markdown("--- ✨ Input patient data to predict the likelihood of Malignant (1) or Benign (0) cancer. ✨ ---")

# Get feature names from the original DataFrame (assuming 'df' is still available)
# If 'df' is not available, you might need to manually list the column names or save them previously
feature_names = X.columns.tolist()

# Sidebar for user input
st.sidebar.header('Patient Data Input 👇')

# Dictionary to store input values
input_data = {}

for feature in feature_names:
    input_data[feature] = st.sidebar.slider(f'Input for {feature}',
                                         float(X[feature].min()),
                                         float(X[feature].max()),
                                         float(X[feature].mean()))

# Create a DataFrame from the input data
input_df = pd.DataFrame([input_data])

st.subheader('User Input Data 📝')
st.write(input_df)

# Scale the input data
input_scaled = scaler.transform(input_df)

# Make prediction
prediction_proba = model.predict(input_scaled)[0][0]
prediction_class = 1 if prediction_proba >= 0.5 else 0

st.subheader('Prediction Result 📊')

if prediction_class == 1:
    st.error(f'The model predicts: Malignant 🚨 (Probability: {prediction_proba:.2f})')
    st.image('https://i.ibb.co/qN3G2yP/malignant.png', caption='Malignant Indication', width=150)
else:
    st.success(f'The model predicts: Benign ✅ (Probability: {prediction_proba:.2f})')
    st.image('https://i.ibb.co/qN3G2yP/benign.png', caption='Benign Indication', width=150)

st.markdown("--- ")
st.write("Disclaimer: This app is for educational purposes only and should not be used for medical advice. Always consult with a healthcare professional. 👩‍⚕️👨‍⚕️")

# To run this Streamlit app, save the code as a .py file (e.g., app.py) and run `streamlit run app.py` in your terminal.
# In Colab, you can use ngrok to expose the Streamlit app.

# Instructions for running in Colab:
# 1. Save the above code to a file named `app.py` in the Colab environment.
#    For example: `%%writefile app.py` at the top of a new cell, followed by the app code.
# 2. Run the following commands in new cells:
#    `!pip install -q ngrok`
#    `!nohup streamlit run app.py &`
#    `!lt -port 8501` (This will give you a public URL for your Streamlit app)

2026-09-08 13:22:19.845 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:19.854 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:20.840 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-09-08 13:22:20.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:20.855 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:20.860 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:20.865 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step


2026-09-08 13:22:22.039 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:22.045 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:22.049 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:22.327 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:22.330 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:22.336 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:22.341 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 13:22:22.346 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [16]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from tensorflow import keras

# Load the scaler and the model
scaler = joblib.load('scaler.pkl')
model = keras.models.load_model('breast_cancer_model.h5')

st.set_page_config(page_title="Breast Cancer Prediction App 🌸", layout="wide")

st.title('Breast Cancer Prediction App 🩺')
st.markdown("--- ✨ Input patient data to predict the likelihood of Malignant (1) or Benign (0) cancer. ✨ ---")

# Get feature names from the original DataFrame (assuming 'df' is still available)
# If 'df' is not available, you might need to manually list the column names or save them previously
# feature_names = X.columns.tolist() # X is not available in the deployed app context
# We need to manually list them or save them with the scaler
# For simplicity, let's derive them from the scaler's features_in_ attribute or a small dummy df
# (Assuming X.columns was saved implicitly by scaler.fit_transform(X_train) and X_train had column names)
# Let's use a placeholder and instruct the user if they want to improve this.

# A more robust way would be to save feature_names explicitly or have a dummy input_df.
# For now, let's assume the order of features is fixed as per X_train.columns.
# For a live app, you'd usually pass feature_names or recreate a dummy DataFrame.

# To make this self-contained, we need feature names here:
# Based on df.info() output, we have 30 float features excluding 'diagnosis'.
# Let's get these from the global X variable used during training setup.
# This code will be written to app.py, so X will not be available. We need to explicitly pass or load it.
# For now, let's hardcode for demonstration, but a real app would load it or have it defined.

# Re-create a dummy X for feature names or ensure X is passed/saved.
# A better way would be to save X.columns to a file or pickle them.
# Let's assume we retrieve the column names from the scaler's fitted features
# Note: StandardScaler itself does not store feature names directly. This requires saving X.columns.
# For this demonstration, we'll retrieve them from the original 'X' DataFrame (which was global in the notebook).
# In a real deployed app, you'd save these feature names (e.g., as a list in a .json or .pkl).

# Placeholder for feature_names, assuming they are available or passed.
# In the actual notebook, 'feature_names = X.columns.tolist()' was already executed.
# Let's just create a dummy df to infer columns if not provided.

# For the Streamlit app to be self-contained and run standalone, 'X' will not be in scope.
# We need to explicitly define feature_names. Since we have 'df' in the notebook context, we can use it.
# Assuming 'X' is available from the notebook's global scope, as it was used to create the scaler.
# In a production app, these feature_names would need to be saved/loaded.

# Let's simulate getting feature names from the original dataset if 'X' isn't directly available
# This is a bit of a workaround for the 'app.py' context not having 'X'.
# A more robust solution would pickle the feature_names list.

# For the purpose of this demo, let's list them directly or from a dummy df if df exists in app.py context.
# Since we are writing the code to 'app.py', we must ensure everything is self-contained.
# We can load a minimal dataset to get feature names if necessary, or pass them explicitly.

# Simplified for demonstration: assuming X (features DataFrame) is somehow accessible for column names
# In a real app, you would load column names from a file or explicitly list them.
# Since X is available in the Python kernel when this `%%writefile` runs, we can reference it.
# If this was a fresh environment for app.py, we'd need a more robust way to get 'feature_names'.

# To make it truly standalone, we should have saved feature_names separately.
# For now, let's directly use the global 'X' from the Colab notebook's environment
# for the writefile command to inject them if it worked like that. Which it doesn't.
# So, I need to make a self-contained feature_names list for the app.py.

# Manual listing of feature names based on the df.info() in previous cells:
feature_names = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean',
                 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean',
                 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se',
                 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
                 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst',
                 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst',
                 'symmetry_worst', 'fractal_dimension_worst']

# Sidebar for user input
st.sidebar.header('Patient Data Input 👇')

# Dictionary to store input values
input_data = {}

# To get min/max/mean for sliders, we need original data statistics. This also needs to be saved/loaded.
# For simplicity, we'll load a small sample of the original data to get these statistics within app.py
# This is less ideal than saving min/max/mean directly, but makes the app self-contained.

# Temporarily load a small DataFrame just to get min/max/mean for the sliders
# This assumes data.csv is accessible to the Streamlit app. For a Kaggle dataset, this is complex.
# A better approach is to save the min/max/mean values to a JSON or PKL file.

# For this demonstration, I'll use the 'X' DataFrame's min/max/mean values directly from the notebook context
# and hardcode them into the `%%writefile` cell. This is not dynamic but works for demonstrating the app.

# To get actual min/max/mean dynamically for the sliders within the Streamlit app itself:
# We would need to load the original 'data.csv' or save these stats.
# Since 'X' is in the current notebook scope, I'll generate the sliders with current 'X' statistics.
# This makes the app.py less portable, but avoids another save/load step for this example.
# Let's save the statistics explicitly.

# Saving stats for feature ranges:
feature_stats = {}
for col in X.columns:
    feature_stats[col] = {
        'min': float(X[col].min()),
        'max': float(X[col].max()),
        'mean': float(X[col].mean())
    }
joblib.dump(feature_stats, 'feature_stats.pkl')

# In app.py, load these stats:
feature_stats = joblib.load('feature_stats.pkl')

for feature in feature_names:
    stats = feature_stats[feature]
    input_data[feature] = st.sidebar.slider(f'Input for {feature}',
                                         stats['min'],
                                         stats['max'],
                                         stats['mean'])

input_df = pd.DataFrame([input_data])

st.subheader('User Input Data 📝')
st.write(input_df)

input_scaled = scaler.transform(input_df)

prediction_proba = model.predict(input_scaled)[0][0]
prediction_class = 1 if prediction_proba >= 0.5 else 0

st.subheader('Prediction Result 📊')

if prediction_class == 1:
    st.error(f'The model predicts: Malignant 🚨 (Probability: {prediction_proba:.2f})')
    st.image('https://i.ibb.co/qN3G2yP/malignant.png', caption='Malignant Indication', width=150)
else:
    st.success(f'The model predicts: Benign ✅ (Probability: {prediction_proba:.2f})')
    st.image('https://i.ibb.co/qN3G2yP/benign.png', caption='Benign Indication', width=150)

st.markdown("--- ")
st.write("Disclaimer: This app is for educational purposes only and should not be used for medical advice. Always consult with a healthcare professional. 👩‍⚕️👨‍⚕️")

Writing app.py


In [17]:
import joblib
# Save feature_stats.pkl for the Streamlit app
feature_stats = {}
for col in X.columns:
    feature_stats[col] = {
        'min': float(X[col].min()),
        'max': float(X[col].max()),
        'mean': float(X[col].mean())
    }
joblib.dump(feature_stats, 'feature_stats.pkl')
print("Feature statistics saved to feature_stats.pkl")

Feature statistics saved to feature_stats.pkl


In [12]:
!pip install --ignore-installed blinker
!pip install streamlit

In [11]:
!pip install streamlit pyngrok

In [13]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [25]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [26]:
# Run the Streamlit app
# This will provide a public URL to access the app
!streamlit run app.py &


2026-09-08 13:25:54.090 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.184.86.203:8501

2026-09-08 13:26:05.136246: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
  Stopping...
